# PULSE GUARD
## Environment Setup

```shell
python3 -m pip install virtualenv
virtualenv --python=python3.14 .venv

source .venv/bin/activate

.venv/bin/python -m pip install ipykernel ipywidgets
```

## PHASE 3: Preprocessing and Tokenisation
Sarkar is the primary dataset. Komati adds Suicidal training examples. SDCNL supports the Depression–Suicidal boundary.

### Install packages
Install the packages used for cleaning, splitting, and tokenisation.

In [1]:
%pip install -q pandas emoji scikit-learn transformers datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setup
Use the raw files from the EDA notebook. Save results under `data/processed`.

In [2]:
from pathlib import Path
import html
import json
import re
import emoji
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
RAW_DIR = NOTEBOOK_DIR / "data/raw"
OUT_DIR = NOTEBOOK_DIR / "data/processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
LABELS = {"Normal": 0, "Depression": 1, "Suicidal": 2}
KOMATI_METHOD = "random"  # "random" or "first"

### Load and standardise
Keep the three Sarkar classes, Suicidal Komati posts, and both SDCNL labels.

In [3]:
sarkar = pd.read_csv(RAW_DIR / "sarkar_2024.csv", usecols=["statement", "status"])
sarkar = sarkar[sarkar["status"].isin(LABELS)].copy()

komati = pd.read_csv(RAW_DIR / "komati_2021.csv", usecols=["text", "class"])
komati = komati.loc[komati["class"] == "suicide", ["text"]].rename(columns={"text": "statement"})
komati["status"] = "Suicidal"

sdcnl = pd.read_csv(RAW_DIR / "sdcnl_2021.csv", usecols=["selftext", "is_suicide"])
sdcnl["is_suicide"] = pd.to_numeric(sdcnl["is_suicide"], errors="raise")

assert sdcnl["is_suicide"].dropna().isin([0, 1]).all()
sdcnl["status"] = sdcnl["is_suicide"].map({0: "Depression", 1: "Suicidal"})
sdcnl = sdcnl.rename(columns={"selftext": "statement"})[["statement", "status"]]

### Compare Komati samples
Compare the first 6,000 available posts with a seeded random sample. Random sampling is the default because file order may affect the first rows. This compares sample composition, not model performance.

In [4]:
komati = komati.dropna(subset=["statement"])
komati = komati[komati["statement"].str.strip().ne("")]

n = min(6000, len(komati))

komati_samples = {
    "first": komati.head(n).copy(),
    "random": komati.sample(n=n, random_state=SEED).copy(),
}

comparison = []
for name, frame in komati_samples.items():
    lengths = frame["statement"].str.split().str.len()
    comparison.append({
        "sample": name, "rows": len(frame),
        "duplicate_texts": frame["statement"].duplicated().sum(),
        "mean_words": lengths.mean(), "median_words": lengths.median(),
        "longest_post": lengths.max(),
    })
    
display(pd.DataFrame(comparison).set_index("sample").round(1))
print("Posts shared by both samples:", len(set(komati_samples["first"].index) & set(komati_samples["random"].index)))

komati = komati_samples[KOMATI_METHOD]

,rows,duplicate_texts,mean_words,median_words,longest_post
sample,,,,,
first,6000,0,197.2,125.0,3580
random,6000,0,198.1,121.0,9684


Posts shared by both samples: 314


### Merge and inspect
Keep source names to preserve each dataset’s role.

In [5]:
df = pd.concat([
    sarkar.assign(source="sarkar"),
    komati.assign(source="komati"),
    sdcnl.assign(source="sdcnl"),
], ignore_index=True)

print("Rows:", len(df))
display(pd.crosstab(df["source"], df["status"]))
display(df[["statement", "status"]].isna().sum())
print("Blank text:", df["statement"].str.strip().eq("").sum())
print("Deleted or removed posts:", df["statement"].str.strip().str.lower().isin(["[deleted]", "[removed]"]).sum())

Rows: 50303


status,Depression,Normal,Suicidal
source,,,
komati,0,0,6000
sarkar,15404,16351,10653
sdcnl,915,0,980


statement    9
status       0
dtype: int64

Blank text: 0
Deleted or removed posts: 0


### Clean text
Decode HTML, turn emoji into text, and replace URLs and Reddit references. Keep stopwords, negation, and punctuation. Drop empty or deleted posts.

In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    if text.strip().lower() in {"[deleted]", "[removed]"}:
        return ""
    text = re.sub(r"\[([^\]]+)\]\(https?://[^\s)]+\)", r"\1", text)
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)
    text = re.sub(r"(?<!\w)/?u/[A-Za-z0-9_-]+", " [USER] ", text)
    text = re.sub(r"(?<!\w)/?r/[A-Za-z0-9_]+", " [SUBREDDIT] ", text)
    text = emoji.demojize(text, delimiters=(" :", ": "))
    return re.sub(r"\s+", " ", text).strip()

before = len(df)
df["clean_text"] = df["statement"].map(clean_text)

df = df[df["clean_text"].ne("") & df["status"].notna()].copy()

print("Empty text or missing labels removed:", before - len(df))
display(df.loc[df["statement"] != df["clean_text"], ["statement", "clean_text"]].head(5))

Empty text or missing labels removed: 9


,statement,clean_text
29,Room cleaning,Room cleaning
52,punyetaaaaaa ™‚,punyetaaaaaa :trade_mark: ‚
80,Can't watch vivo for 4 more hours otw pls ™ƒ,Can't watch vivo for 4 more hours otw pls :tra...
140,"homecoming holiday, friends, miss my extra money","homecoming holiday, friends, miss my extra money"
145,Thank you so much for those who want to follow...,Thank you so much for those who want to follow...


### Remove conflicts and duplicates
Compare cleaned text without case differences. Save conflicting labels for review and exclude them. Keep one copy of repeated text, preferring Sarkar, then SDCNL, then Komati. This does not detect all near-duplicates.

In [7]:
df["text_key"] = df["clean_text"].str.lower()

conflict = df.groupby("text_key")["status"].transform("nunique").gt(1)

df.loc[conflict].to_csv(OUT_DIR / "label_conflicts.csv", index=False)
print("Conflicting rows excluded:", conflict.sum())

df = df.loc[~conflict].copy()

before = len(df)

df["priority"] = df["source"].map({"sarkar": 0, "sdcnl": 1, "komati": 2})
df = df.sort_values("priority", kind="stable").drop_duplicates("text_key").drop(columns="priority")
df = df.reset_index(drop=True)

print("Duplicate rows removed:", before - len(df))
display(pd.crosstab(df["source"], df["status"]))

Conflicting rows excluded: 111
Duplicate rows removed: 639


status,Depression,Normal,Suicidal
source,,,
komati,0,0,5999
sarkar,15081,16019,10629
sdcnl,907,0,909


### Check cleaned lengths
Compare word counts after cleaning. The Komati sample may now contain fewer than 6,000 posts.

In [8]:
word_lengths = df["clean_text"].str.split().str.len()
display(word_lengths.groupby(df["status"]).agg(["count", "mean", "median", "max"]).round(1))

,count,mean,median,max
status,,,,
Depression,15988,169.1,113.0,4239
Normal,16019,17.5,10.0,255
Suicidal,17537,165.2,101.0,9684


### Split the data
Split Sarkar 80/10/10 for training, validation, and testing. Add Komati only to training. Use 70% of SDCNL for training and hold out 30% for boundary evaluation. This is a new split of SDCNL’s combined file.

In [9]:
sarkar = df[df["source"] == "sarkar"]
komati = df[df["source"] == "komati"]
sdcnl = df[df["source"] == "sdcnl"]

primary = pd.concat([sarkar, komati], ignore_index=True)

train, test = train_test_split(
    primary,
    test_size=0.2,
    stratify=primary["status"], 
    random_state=SEED,
)

sdcnl_train, sdcnl_test = train_test_split(
    sdcnl, test_size=0.2, stratify=sdcnl["status"], random_state=SEED
)

train = pd.concat([train, sdcnl_train], ignore_index=True)
test = pd.concat([test, sdcnl_test], ignore_index=True)

splits = {"train": train, "test": test}

final_df = pd.concat([frame.assign(split=name) for name, frame in splits.items()], ignore_index=True)
final_df["labels"] = final_df["status"].map(LABELS)

assert not final_df["text_key"].duplicated().any()
assert final_df["labels"].notna().all()

display(pd.crosstab([final_df["split"], final_df["source"]], final_df["status"]))

final_df.to_csv(OUT_DIR / "clean_dataset.csv", index=False)

status        Depression  Normal  Suicidal
split source                              
test  komati           0       0      1216
      sarkar        3016    3204      2110
      sdcnl          182       0       182
train komati           0       0      4783
      sarkar       12065   12815      8519
      sdcnl          725       0       727

## PHASE 4: Tokenisation
### Load MentalBERT’s tokenizer
Accept access on the [model page](https://huggingface.co/mental/mental-bert-base-uncased), then run `hf auth login` in your activated terminal if required. Do not put tokens in this notebook.

In [12]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

MODEL_NAME = "mental/mental-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--mental--mental-bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### Check training token lengths
Count tokens before truncation. Show how many training posts exceed each limit. The chosen limit keeps only the start of longer posts.

In [13]:
train_texts = final_df.loc[final_df["split"] == "train", "clean_text"].tolist()

lengths = []
for start in range(0, len(train_texts), 256):
    encoded = tokenizer(train_texts[start:start + 256], truncation=False, padding=False)
    lengths.extend(len(ids) for ids in encoded["input_ids"])
lengths = pd.Series(lengths)

display(lengths.describe())
display(pd.DataFrame({
    "token_limit": [128, 256, 512],
    "training_posts_truncated": [(lengths > limit).sum() for limit in [128, 256, 512]],
}))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1777 > 512). Running this sequence through the model will result in indexing errors


count    39634.000000
mean       141.192537
std        219.589419
min          3.000000
25%         22.000000
50%         72.000000
75%        179.000000
max      11624.000000
dtype: float64

,token_limit,training_posts_truncated
0,128,13468
1,256,6335
2,512,1829


### Tokenise and save
Save token IDs, attention masks, and labels for each split. Pad batches during model training with `DataCollatorWithPadding(tokenizer)`. The cleaned CSV retains source and split details.

In [14]:
MAX_LENGTH = 512

dataset = DatasetDict({
    name: Dataset.from_pandas(
        final_df.loc[final_df["split"] == name, ["clean_text", "labels"]],
        preserve_index=False,
    ) for name in splits
})

def tokenise(batch):
    return tokenizer(batch["clean_text"], truncation=True, max_length=MAX_LENGTH, padding=False)

tokenised = dataset.map(tokenise, batched=True, remove_columns=["clean_text"])

assert all(len(ids) <= MAX_LENGTH for part in tokenised.values() for ids in part["input_ids"])
tokenised.save_to_disk(str(OUT_DIR / "tokenised"))
tokenizer.save_pretrained(OUT_DIR / "tokenizer")

settings = {"model": MODEL_NAME, "max_length": MAX_LENGTH, "label2id": LABELS,
            "seed": SEED, "komati_method": KOMATI_METHOD}
(OUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2))
print(tokenised)

Map:   0%|          | 0/39634 [00:00<?, ? examples/s]

Map:   0%|          | 0/9910 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/39634 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9910 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 39634
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9910
    })
})


[Hugging Face: padding and truncation](https://huggingface.co/docs/transformers/pad_truncation).